In [ ]:
import warnings
warnings.filterwarnings("ignore")
import scanpy as sc
import os
import torch
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import NearestNeighbors
from scipy.linalg import sqrtm 
import numpy as np

import pandas as pd

In [ ]:
import sys
import os
project_root = os.path.abspath('../../')
if project_root not in sys.path:
    sys.path.append(project_root)
print(f"Project root added to sys.path: {project_root}")

In [ ]:
# import spatial multi-omics data
dataset_file = '/mnt/msg/1-PRISM/0-dataset/1-human lymphoid organs/lymph/slice1/'

adata_omics1 = sc.read_h5ad(dataset_file + 'adata_RNA.h5ad')
adata_omics2 = sc.read_h5ad(dataset_file + 'adata_ADT.h5ad')
adata_omics1.var_names_make_unique()
adata_omics2.var_names_make_unique()

In [ ]:
# Raw data: RNA
adata_omics1

In [ ]:
# Number of HVGs: Defined based on the actual data captured, and can be referred to in "Haviv et al. Nat Biotech 2024".
# (Set to 64 for faster calculation)
hvgs = 64 

In [ ]:
from PRISM.preprocess import clr_normalize_each_cell, pca
#1 Data preprocessing: RNA
sc.pp.filter_genes(adata_omics1, min_cells=10)
sc.pp.highly_variable_genes(adata_omics1, flavor="seurat_v3", n_top_genes=hvgs) # Here you can choose HVGs based on your data situation.
sc.pp.normalize_total(adata_omics1, target_sum=1e4)
sc.pp.log1p(adata_omics1)
# adata_omics1 = clr_normalize_each_cell(adata_omics1)
adata_omics1 = adata_omics1[:, adata_omics1.var['highly_variable']]

#2 Data preprocessing: Protein
sc.pp.normalize_total(adata_omics2, target_sum=1e4)
sc.pp.log1p(adata_omics2)

In [ ]:
# Processed data: RNA
adata_omics1

In [ ]:
# Value range
print(adata_omics1.X.min(), adata_omics1.X.max())
print(adata_omics2.X.min(), adata_omics2.X.max())

In [ ]:
from PRISM.covet.covet_aot import CovetConfig, AotGraphConfig, compute_covet, build_aot_knn_graph

# =========================
# Device selection examples (same logic for COVET and AOT):
#   - device=None / knn_device=None: auto (cuda:0 if available else cpu)
#   - "cpu": force CPU
#   - "cuda" or "gpu": default cuda:0
#   - "cuda:1" or "gpu:1": choose specific GPU index (fallback to cuda:0 if out of range)
# =========================

# Choose device here (edit only this line if you want):
covet_device = "cuda:1"     # e.g. None / "cpu" / "c    uda" / "cuda:1" / 1
aot_knn_device = "cuda:1"   # e.g. None / "cpu" / "cuda" / "cuda:1" / 1


# 1) COVET：Prints per 10000
# If the memory used for running the single-cell data exceeds the limit, we recommend setting use_chunking=True.
adata_omics1 = compute_covet(
    adata_omics1,
    CovetConfig(
        k_spatial=6, # k_cell=8 and k_spot=6.
        genes="hvg", # Here we recommend using highly variable genes (HVGs) for computing COVET, which can be selected based on your data situation (you can choose 'all' or 'hvg').
        n_hvg=hvgs,
        device=covet_device, # Strongly recommend using GPU for computing. (supports: None/"cpu"/"cuda"/"cuda:idx"/int idx)
        use_chunking=False, 
        chunk_size=3000,
        log_every=3000,
        store_prefix="covet",
        verbose=True
    )
)
# The output result of COVET is: adata_omics1.obsm["covet_sqrt_ut"]

In [ ]:
# 2) COVET_AOT_Distance
k_env_value = adata_omics1.n_obs-1 # Here k_env_value is the number of cells/spots
adata_omics1 = build_aot_knn_graph(
    adata_omics1,
    AotGraphConfig(
        covet_ut_key="covet_sqrt_ut",
        k_env=k_env_value,
        knn_backend="torch",
        knn_device=aot_knn_device, 
        use_chunking=False, 
        chunk_size=3000,
        log_every=3000,
        store_prefix="aot",
        verbose=True
    )
)
# The output result of AOT_Distances is: adata_omics1.obsp["aot_distances"]  (csr Sparse distance Matrix)

In [ ]:
import os
import numpy as np
import scipy.sparse as sp

print("The current working directory:", os.getcwd())

# Read the sparse distance matrix (SQUARED distances, paper definition)
aot_dist2 = adata_omics1.obsp["aot_distances"]  # csr_matrix
assert sp.isspmatrix(aot_dist2), "adata_omics1.obsp['aot_distances'] is scipy.sparse matrix"

# Save_dir
save_dir = "../2-Save_file"
os.makedirs(save_dir, exist_ok=True)
save_file = os.path.join(save_dir, "S1-AOT_distance.npz")
sp.save_npz(save_file, aot_dist2)

# Load back
loaded_aot_dist = sp.load_npz(save_file)
print("import the shape of matrix:", loaded_aot_dist.shape, "nnz=", loaded_aot_dist .nnz)

# Option 1: display a small block
block2 = loaded_aot_dist[:5, :5].toarray().astype(np.float32)
block = np.sqrt(np.maximum(block2, 0.0))
print("\nDisplayed distance (sqrt) matrix (first 5x5 block):")
print(block)

### Validation (use pcc,spcc and rmse)

In [ ]:
# Import the function from Validation.py in the PRISM folder
from PRISM.Validation import compute_metrics_each_pair

# Distance matrix is usually sparse CSR
distance_matrix = adata_omics1.obsp["aot_distances"]

# Call the function from Validation.py
rmse_values, pcc_values, spcc_values, details = compute_metrics_each_pair(
    adata_omics2,
    distance_matrix,
    top_n=5,
    verbose=False
)

# Print the top_n neighbors and corresponding distances for the first 3 spots
for detail in details[:3]:
    i = detail["spot_index"]
    neigh = detail["similar_spots"]
    dists2 = detail["similar_dists"]  
    dists = np.sqrt(np.maximum(dists2, 0))
    print(f"\nSpot {i} top-{len(neigh)} neighbors:")
    for j, d in zip(neigh, dists):
        print(f"  -> neighbor {j:5d}  dist={d:.6g}")

# Print overall metrics
print("\nOverall Metrics:")
print(f"Mean RMSE: {np.nanmean(rmse_values):.4f}")
print(f"Mean PCC : {np.nanmean(pcc_values):.4f}")
print(f"Mean SPCC: {np.nanmean(spcc_values):.4f}")